In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# トランスパイルでよく使われるパラメーター

<details>
<summary><b>パッケージバージョン</b></summary>

このページのコードは、以下の要件を使って開発されました。
これらのバージョン以降の使用を推奨します。

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

このページでは、ローカルトランスパイルでよく使われるパラメーターのいくつかを説明します。これらのパラメーターは、[`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) または [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile) への引数を使って設定します。

<span id="approx-degree"></span>
## 近似度
近似度（approximation degree）を使用して、出力回路を目的の（入力）回路にどれだけ近づけたいかを指定できます。これは (0.0 ～ 1.0) の範囲の浮動小数点数で、0.0 が最大近似、1.0（デフォルト）が近似なしを意味します。値を小さくすると、出力精度を犠牲にして実行しやすくなります（つまり、ゲート数が少なくなります）。デフォルト値は 1.0 です。

2量子ビットユニタリー合成（全レベルの初期ステージおよび最適化レベル 3 での最適化ステージで使用）では、この値が出力分解の目標忠実度を指定します。つまり、回路の行列表現が離散ゲートに変換される際にどれだけの誤差が生じるかを示します。近似度の値が小さい（近似が大きい）場合、合成から得られる出力回路は入力行列との差が大きくなりますが、ゲート数も少なくなります（任意の 2 量子ビット演算は最大 3 つの CX ゲートで完全に分解できるため）、実行しやすくなります。

近似度が 1.0 未満の場合、1 つまたは 2 つの CX ゲートで回路が合成される可能性があり、ハードウェアからの誤差は少なくなりますが、近似による誤差が増えます。CX は誤差の観点で最もコストの高いゲートなので、合成の忠実度を犠牲にしてでも CX の数を減らすことが有益な場合があります（この手法は IBM&reg; デバイスでの量子ボリューム向上に使用されました: [Validating quantum computers using randomized model circuits](https://arxiv.org/abs/1811.12926)）。

例として、初期ステージで合成されるランダムな 2 量子ビット `UnitaryGate` を生成します。`approximation_degree` を 1.0 未満に設定すると、近似回路が生成される場合があります。また、合成メソッドが近似合成に使用できるゲートを認識できるよう、`basis_gates` も指定する必要があります。

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import random_unitary
from qiskit.transpiler import generate_preset_pass_manager

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qubits = QuantumRegister(2, name="q")
qc = QuantumCircuit(qubits)
qc.append(rand_U, qubits)
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    approximation_degree=0.85,
    basis_gates=["sx", "rz", "cx"],
)
approx_qc = pass_manager.run(qc)
print(approx_qc.count_ops()["cx"])

2


This yields an output of `2` because the approximation requires fewer CX gates.

<span id="seed"></span>
## Random number generator seed

Some parts of the transpiler are stochastic, so repeated transpilation runs may return different results. To obtain a reproducible result, you can set the seed for the pseudorandom number generator using the `seed_transpiler` argument. Repeated runs using the same seed will return the same results.

Example:

In [2]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, seed_transpiler=11, basis_gates=["sx", "rz", "cx"]
)
optimized_1 = pass_manager.run(qc)
optimized_1.draw("mpl")

<Image src="../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg" alt="Output of the previous code cell" />

近似により CX ゲートの数が少なくて済むため、出力は `2` になります。

<span id="seed"></span>
## 乱数ジェネレーターのシード
Transpilerの一部は確率的に動作するため、トランスパイルを繰り返し実行すると異なる結果が返ることがあります。再現可能な結果を得るには、`seed_transpiler` 引数を使って疑似乱数ジェネレーターのシードを設定できます。同じシードを使って繰り返し実行すると、同じ結果が返ります。

例：

In [3]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler import Layout

backend = FakeSherbrooke()

a, b = qubits
initial_layout = Layout({a: 5, b: 6})

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg)

<span id="init-layout"></span>
## 初期レイアウト
トランスパイル前、回路に含まれる Qubit は仮想 Qubit であり、ターゲット Backend の物理 Qubit に必ずしも対応していません。`initial_layout` 引数を使って、仮想 Qubit から物理 Qubit への初期マッピングを指定できます。ただし、Transpilerがスワップゲートやその他の手段で Qubit を並べ替えることがあるため、最終的な Qubit レイアウトは初期レイアウトと異なる場合があります。

以下の例では、[`Layout`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Layout) オブジェクトを作成して、[`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke) モック Backend の初期レイアウトを構築します。このレイアウトでは、回路の最初の Qubit を Sherbrooke の Qubit 5 に、回路の 2 番目の Qubit を Sherbrooke の Qubit 6 にマッピングします。物理 Qubit は常に整数で表されることに注意してください。

In [4]:
initial_layout = [5, 6]

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg)

Layout オブジェクトを指定する方法に加えて、整数のリストを渡すこともできます。リストの $i$ 番目の要素に、$i$ 番目の Qubit がマッピングされるべき物理 Qubit を指定します。例：

In [5]:
from qiskit.visualization import plot_error_map

plot_error_map(backend, figsize=(30, 24))

<Image src="../docs/images/guides/common-parameters/extracted-outputs/8df57c6a-1ff4-4d58-9b7e-4378452c3025-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg)

[`plot_error_map`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.plot_error_map) 関数を使って、誤差情報と物理 Qubit のラベルが付いたデバイスグラフの図を生成できます。また、[Compute resources](https://quantum.cloud.ibm.com/computers) ページで同様の図を確認することもできます。